In [ ]:
# ---- Data download/Dataframe creation ----
from arcgis.gis import GIS
import pandas as pd

# Connect to AGOL (uses your Pro login if you're signed in)
gis = GIS("home")

# Item IDs from AGOL
editlayer_item_id = "945c7968f26449988306eb3a98935c94"
mailing_item_id = "41af864d040d46578b8eefbd785a27d6"

# Load layers
editlayer_item = gis.content.get(editlayer_item_id)
mailing_item = gis.content.get(mailing_item_id)

editlayer_layer = editlayer_item.layers[0]
mailing_layer = mailing_item.layers[0]

# Query all records into DataFrames
editlayer_df = editlayer_layer.query(where="1=1", as_df=True)
mailing_df = mailing_layer.query(where="1=1", as_df=True)

# --- Clean APN fields (VERY IMPORTANT) ---
editlayer_df["APN"] = editlayer_df["APN"].astype(str).str.replace("-", "").str.strip()
mailing_df["APN"] = mailing_df["APN"].astype(str).str.replace("-", "").str.strip()

def build_address(row):
    parts = [
        row.get("MailAddress", ""),
        row.get("MailCity", ""),
        row.get("MailZip", "")
    ]
    return ", ".join([str(p).strip() for p in parts if pd.notnull(p) and str(p).strip() != ""])

mailing_df["MailingAddress"] = mailing_df.apply(build_address, axis=1)

# --- Join ---
joined_df = editlayer_df.merge(
    mailing_df,
    on="APN",
    how="left"   # change to "left" if you want all editlayer
)

# --- Create master file ---

# Output Excel path
output_excel = r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - Compliance\2026 Compliance Document Creation\MasterFile.csv"

# Export
joined_df.to_csv(output_excel, index=False)

print("Excel file created at:", output_excel)

# --- Access the request layer ---
item = gis.content.get("b6a36b093b734022bece2106ce37dd7c")
layer = item.layers[0]

requestlayer = layer.query(
    where="1=1",
    as_df=True,
    return_all_records=True
)

#Rename APN column to merge on
requestlayer = requestlayer.rename(columns={"apn": "APN"})

#format the CreationDate column to filter by month, also normalize APNs
requestlayer["CreationDate"] = pd.to_datetime(requestlayer["CreationDate"], errors="coerce")

requestlayer["APN"] = requestlayer["APN"].astype(str).str.replace("-", "").str.strip()

# Select APNs from this month
# AND only where PROPERTYSTATUS = "Compliant"

month_apns = requestlayer.loc[
    (
        requestlayer["CreationDate"].dt.month == 4
    )
    &
    (
        requestlayer["PROPERTYSTATUS"]
        .astype(str)
        .str.strip()
        .eq("Compliant")
    ),
    "APN"
]

# Remove the month entries from the master
clean_df = joined_df[
    ~joined_df["APN"].isin(month_apns)
]

# Keep only rows where PROPERTYSTATUS is "Compliant"
clean_df = clean_df[
    clean_df["PROPERTYSTATUS"]
    .astype(str)
    .str.strip()
    .eq("Compliant")
]

# Keep only the desired columns
clean_df = clean_df[
    ["SCR_INSPDATENEWEST", "APN", "PARCELSITEADDRESS", "MailAddress", "MailCity", "MailZip"]
]

#filer out properties requested in this month from the master file
filtered_joined_df = joined_df[
    joined_df["APN"].isin(month_apns)
]

In [ ]:
# ---- File Export ----
#clean_excel = r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - Compliance\2026 Compliance Document Creation\CleanFile.csv"
month_excel = r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - Compliance\2026 Compliance Document Creation\TODAY\monthEmailCompliance.csv"

# Export
#clean_df.to_csv(clean_excel, index=False)
filtered_joined_df.to_csv(month_excel, index=False)

In [ ]:
# ---- Can use this chunk interchangebly [when stopping/restarting the script]; BE SURE TO CHECK WHICH FILE IS BEING READ IN!!!!----
import pandas as pd

clean_df = pd.read_csv(
    r"C:\Users\e705941\OneDrive - County of Los Angeles"
    r"\Defensible Space Unit - 2026 Defensible Space Cycle"
    r"\Compliance\2026 Compliance Document Creation\FinalFiltered.csv"
)

In [ ]:
import pandas as pd
duplicates_df = pd.read_csv(
    r"C:\Users\e705941\OneDrive - County of Los Angeles"
    r"\Defensible Space Unit - 2026 Defensible Space Cycle"
    r"\Compliance\2026 Compliance Document Creation\joinwparcels_EDITED.csv",
    #usecols=["APN", "APN_1", "PARCELSITEADDRESS", "SCR_INSPDATENEWEST","MailAddress","MailCity","MailZip"]
)

clean_df = duplicates_df[
    duplicates_df["APN_1"].duplicated(keep=False)
]


In [ ]:
print(clean_df)

In [ ]:
# --- Full batch with PDF conversion ---

from mailmerge import MailMerge
from docx2pdf import convert

import pandas as pd
import os
import math

#IF YOU CHANGED THE FILE TO BE READ IN DOUBLE CHECK THIS PATH!!
os.chdir(
    r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Compliance\2026 Compliance Document Creation"
)

# -----------------------------------
# COPY FULL DATAFRAME
# -----------------------------------

full_df = clean_df.copy()

# -----------------------------------
# FORMAT DATE COLUMN
# -----------------------------------

full_df["SCR_INSPDATENEWEST"] = (
    pd.to_datetime(
        full_df["SCR_INSPDATENEWEST"],
        errors="coerce"
    )
    .dt.strftime("%m/%d/%Y")
)

# -----------------------------------
# CLEAN APN
# -----------------------------------

full_df["APN"] = (
    full_df["APN"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^0-9]", "", regex=True)
    .str.zfill(9)
    .str.replace(
        r"(\d{4})(\d{3})(\d{3})",
        r"\1-\2-\3",
        regex=True
    )
)

# -----------------------------------
# CLEAN ZIP
# -----------------------------------

full_df["MailZip"] = (
    full_df["MailZip"]
    .astype(str)
    .str.extract(r"(\d{5})", expand=False)
)

# -----------------------------------
# CONVERT ALL VALUES TO STRING
# -----------------------------------

full_df = full_df.fillna("")
full_df = full_df.astype(str)
full_df = full_df.replace("nan", "")

# -----------------------------------
# BATCH SETTINGS
# -----------------------------------

batch_size = 50
total_rows = len(full_df)

num_batches = math.ceil(
    total_rows / batch_size
)

print(f"Total rows: {total_rows}")
print(f"Creating {num_batches} batch files...")

# -----------------------------------
# OPTIONAL RESTART POINT
# -----------------------------------

start_batch = 0

# -----------------------------------
# PROCESS BATCHES
# -----------------------------------

for batch_num in range(
    start_batch,
    num_batches
):

    start_row = batch_num * batch_size
    end_row = start_row + batch_size

    batch_df = full_df.iloc[
        start_row:end_row
    ].copy()

    print(
        f"Processing batch {batch_num + 1} "
        f"({start_row:,} to "
        f"{min(end_row, total_rows):,})"
    )

    # -----------------------------------
    # OPEN TEMPLATE
    # -----------------------------------

    document = MailMerge(
        "Compliance Letter Power Automate ----.docx"
    )

    # -----------------------------------
    # CONVERT ROWS TO RECORDS
    # -----------------------------------

    records = batch_df.to_dict(
        orient="records"
    )

    # -----------------------------------
    # MERGE RECORDS
    # -----------------------------------

    document.merge_templates(
        records,
        separator="continuous_section"
    )

    # -----------------------------------
    # FILE NAMES
    # -----------------------------------

    docx_filename = (
        f"mail_merge_output_batch_"
        f"{batch_num + 1}.docx"
    )

    pdf_filename = (
        f"mail_merge_output_batch_"
        f"{batch_num + 1}.pdf"
    )

    # -----------------------------------
    # SAVE DOCX
    # -----------------------------------

    print("Writing DOCX...")

    document.write(docx_filename)

    document.close()

    print(f"Saved DOCX: {docx_filename}")

    # -----------------------------------
    # CONVERT TO PDF
    # -----------------------------------

    try:

        print("Converting to PDF...")

        convert(
            docx_filename,
            pdf_filename
        )

        print(f"Saved PDF: {pdf_filename}")

        # -----------------------------------
        # OPTIONAL:
        # DELETE DOCX AFTER CONVERSION
        # -----------------------------------

        os.remove(docx_filename)

        # -----------------------------------
        # OPTIONAL:
        # CLOSE WORD AFTER EACH BATCH
        # -----------------------------------

        os.system(
            "taskkill /f /im WINWORD.EXE"
        )

    except Exception as e:

        print(
            f"PDF conversion failed "
            f"for batch {batch_num + 1}"
        )

        print(e)

print("All batches completed.")

In [ ]:
#----In case of stop and restart----

from mailmerge import MailMerge
from docx2pdf import convert
import pandas as pd
import os
import math

os.chdir(
    r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Compliance\2026 Compliance Document Creation"
)

# -----------------------------------
# COPY FULL DATAFRAME
# -----------------------------------

full_df = clean_df.copy()

# -----------------------------------
# FORMAT DATE COLUMN
# -----------------------------------

full_df["SCR_INSPDATENEWEST"] = (
    pd.to_datetime(full_df["SCR_INSPDATENEWEST"], errors="coerce")
    .dt.strftime("%m/%d/%Y")
)

# -----------------------------------
# CLEAN APN
# -----------------------------------

full_df["APN"] = (
    full_df["APN"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^0-9]", "", regex=True)
    .str.zfill(9)
    .str.replace(
        r"(\d{4})(\d{3})(\d{3})",
        r"\1-\2-\3",
        regex=True
    )
)

# -----------------------------------
# CLEAN ZIP
# -----------------------------------

full_df["MailZip"] = (
    full_df["MailZip"]
    .astype(str)
    .str.extract(r"(\d{5})", expand=False)
)

# -----------------------------------
# CONVERT ALL VALUES TO STRING
# -----------------------------------

full_df = full_df.fillna("")
full_df = full_df.astype(str)
full_df = full_df.replace("nan", "")

# -----------------------------------
# BATCH SETTINGS
# -----------------------------------

batch_size = 50
total_rows = len(full_df)
num_batches = math.ceil(total_rows / batch_size)

print(f"Total rows: {total_rows}")
print(f"Creating {num_batches} batch files...")

# -----------------------------------
# RESTART POINT
# -----------------------------------
# Batch numbering is zero-based internally
# Batch 12 = rows 5500-5999
# -----------------------------------

start_batch = 474

# -----------------------------------
# PROCESS BATCHES
# -----------------------------------

for batch_num in range(start_batch, num_batches):

    start_row = batch_num * batch_size
    end_row = start_row + batch_size

    batch_df = full_df.iloc[start_row:end_row].copy()

    print(
        f"Processing batch {batch_num + 1} "
        f"({start_row:,} to {min(end_row, total_rows):,})"
    )

    # Open template fresh each batch
    document = MailMerge(
        "Compliance Letter Power Automate ----.docx"
    )

    # Convert rows to records
    records = batch_df.to_dict(orient="records")

    # Merge records
    document.merge_templates(
        records,
        separator="continuous_section"
    )

    # Temporary DOCX filename
    temp_docx = (
        f"mail_merge_output_batch_{batch_num + 1}.docx"
    )

    # Final PDF filename
    output_pdf = (
        f"mail_merge_output_batch_{batch_num + 1}.pdf"
    )

    # Save temporary DOCX
    document.write(temp_docx)

    # Close document
    document.close()

    # Convert to PDF
    convert(temp_docx, output_pdf)

    # OPTIONAL:
    # Delete temp DOCX after PDF conversion
    os.remove(temp_docx)

    print(f"Saved PDF: {output_pdf}")

print("All batches completed.")

In [ ]:
#----In case of stop and restart; START AND STOP POINT----

from mailmerge import MailMerge
import pandas as pd
import os
import math
import time
import pythoncom
import win32com.client
import pywintypes
from pathlib import Path

# ==========================================
# PATHS
# ==========================================

base_folder = Path(
    r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Compliance\2026 Compliance Document Creation"
)

os.chdir(base_folder)

recovery_folder = base_folder / "RECOVERY_BATCH_10_REBUILD"
recovery_folder.mkdir(exist_ok=True)

temp_folder = base_folder / "TEMP_WORD_FILES"
temp_folder.mkdir(exist_ok=True)

# ==========================================
# DATA PREP
# ==========================================

full_df = clean_df.copy()

full_df["SCR_INSPDATENEWEST"] = (
    pd.to_datetime(full_df["SCR_INSPDATENEWEST"], errors="coerce")
    .dt.strftime("%m/%d/%Y")
)

full_df["APN"] = (
    full_df["APN"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^0-9]", "", regex=True)
    .str.zfill(9)
    .str.replace(r"(\d{4})(\d{3})(\d{3})", r"\1-\2-\3", regex=True)
)

full_df["MailZip"] = (
    full_df["MailZip"]
    .astype(str)
    .str.extract(r"(\d{5})", expand=False)
)

full_df = full_df.fillna("")
full_df = full_df.astype(str)
full_df = full_df.replace("nan", "")

# ==========================================
# SETTINGS
# ==========================================

batch_size = 10
total_rows = len(full_df)
num_batches = math.ceil(total_rows / batch_size)

print(f"Total rows: {total_rows}")
print(f"Total batches: {num_batches}")

start_batch = 523
end_batch = 957

print(f"Recovering batches {start_batch} → {end_batch - 1}")

# ==========================================
# SAFE WORD EXPORT FUNCTION
# ==========================================

def convert_to_pdf(docx_path, pdf_path):
    pythoncom.CoInitialize()

    word = win32com.client.DispatchEx("Word.Application")
    word.Visible = False

    try:
        doc = word.Documents.Open(str(docx_path))

        success = False

        for attempt in range(3):
            try:
                doc.ExportAsFixedFormat(str(pdf_path), 17)
                success = True
                break
            except pywintypes.com_error:
                print(f"Word busy, retry {attempt+1}/3 for {docx_path.name}")
                time.sleep(3)

        doc.Close(False)

        return success

    finally:
        word.Quit()

# ==========================================
# MAIN LOOP
# ==========================================

for batch_num in range(start_batch, end_batch):

    start_row = batch_num * batch_size
    end_row = start_row + batch_size

    batch_df = full_df.iloc[start_row:end_row].copy()

    if batch_df.empty:
        print(f"Skipping empty batch {batch_num + 1}")
        continue

    print(
        f"\nProcessing batch {batch_num + 1} "
        f"({start_row:,} to {min(end_row, total_rows):,})"
    )

    try:
        document = MailMerge("Compliance Letter Power Automate ----.docx")

        records = batch_df.to_dict(orient="records")

        document.merge_templates(
            records,
            separator="continuous_section"
        )

        output_pdf = recovery_folder / f"mail_merge_output_batch_{batch_num + 1}.pdf"
        temp_docx = temp_folder / f"temp_{batch_num + 1}.docx"

        # Skip if already done
        if output_pdf.exists():
            print(f"Skipping existing: {output_pdf.name}")
            document.close()
            continue

        document.write(str(temp_docx))
        document.close()

        # Convert
        ok = convert_to_pdf(temp_docx, output_pdf)

        if ok:
            print(f"Saved: {output_pdf.name}")
        else:
            print(f"FAILED conversion: batch {batch_num + 1}")

        # safer delete
        time.sleep(1.5)

        try:
            if temp_docx.exists():
                temp_docx.unlink()
        except PermissionError:
            print(f"Could not delete temp file: {temp_docx.name}")

        # reduce Word COM buildup pressure
        time.sleep(1)

    except Exception as e:
        print(f"ERROR in batch {batch_num + 1}: {e}")

    finally:
        pythoncom.CoUninitialize()

print("\nRECOVERY COMPLETE")

In [ ]:
#---to kill word within python---
import os

os.system("taskkill /f /im WINWORD.EXE")

In [ ]:
# ---- CODE TO CONVERT LEFTOVER WORD DOCS ----
from pathlib import Path
import win32com.client

folder = Path(
    r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Compliance\2026 Compliance Document Creation"
)

# PDF format constant for Word
wdFormatPDF = 17

word = win32com.client.Dispatch("Word.Application")
word.Visible = False

converted = 0
deleted = 0
errors = []

try:
    docx_files = sorted(folder.glob("mail_merge_output_batch_*.docx"))

    for docx_file in docx_files:
        pdf_file = docx_file.with_suffix(".pdf")

        try:
            # Create PDF if missing
            if not pdf_file.exists():
                print(f"Converting: {docx_file.name}")

                doc = word.Documents.Open(str(docx_file))
                doc.SaveAs(str(pdf_file), FileFormat=wdFormatPDF)
                doc.Close(False)

                converted += 1

            # Delete DOCX if PDF exists
            if pdf_file.exists():
                docx_file.unlink()
                print(f"Deleted: {docx_file.name}")
                deleted += 1

        except Exception as e:
            errors.append((docx_file.name, str(e)))
            print(f"ERROR: {docx_file.name} -> {e}")

finally:
    word.Quit()

print("\n--- Summary ---")
print(f"Converted to PDF: {converted}")
print(f"Deleted DOCX files: {deleted}")
print(f"Errors: {len(errors)}")

if errors:
    print("\nError Details:")
    for filename, error in errors:
        print(f"{filename}: {error}")

In [ ]:
# ---- Keyword/APN search through all PDF batches ----

from pathlib import Path
import fitz  # PyMuPDF
import pandas as pd

# ==========================================
# FOLDERS
# ==========================================

folders = [
    Path(
        r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Compliance\2026 Compliance Document Creation\April\Mailed Letters\Ready to Print"
    )
#    Path(
#        r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Compliance\2026 Compliance Document Creation\Archive\5_29_merged_batches\PRINT READY"
#    )
]

# ==========================================
# SEARCH TERMS
# ==========================================

search_terms = ["510 E. Viewcrest Drive"]

# ==========================================
# SEARCH
# ==========================================

results = []

for folder in folders:

    pdf_files = sorted(folder.glob("*.pdf"))

    print(f"Scanning {len(pdf_files)} PDFs in {folder.name}")

    for pdf_file in pdf_files:

        try:
            pdf = fitz.open(pdf_file)

            for page_num in range(len(pdf)):

                text = pdf[page_num].get_text()

                text_lower = text.lower()

                for term in search_terms:

                    if term in text_lower:

                        idx = text_lower.find(term)

                        snippet_start = max(0, idx - 100)
                        snippet_end = min(len(text), idx + 100)

                        snippet = text[snippet_start:snippet_end].replace("\n", " ")

                        results.append({
                            "Folder": folder.name,
                            "PDF": pdf_file.name,
                            "Page": page_num + 1,
                            "Keyword": term,
                            "Snippet": snippet
                        })

            pdf.close()

        except Exception as e:
            print(f"Error reading {pdf_file.name}: {e}")

# ==========================================
# RESULTS
# ==========================================

results_df = pd.DataFrame(results)

print("\n================================")
print(f"Matches found: {len(results_df)}")
print("================================")

if not results_df.empty:
    print(results_df[["PDF", "Page", "Keyword"]])

#    output_file = (
#        folders[0] / "APN_Search.xlsx"
#    )

#    results_df.to_excel(output_file, index=False)
#
#    print(f"\nSaved results to:\n{output_file}")
else:
    print("No matches found.")

In [ ]:
# ---- Page removal based on Keyword/APN ----
from pathlib import Path
import pandas as pd
import fitz  # PyMuPDF

# ==========================================
# PATHS
# ==========================================

base_folder = Path(
    r"C:\Users\e705941\OneDrive - County of Los Angeles\Defensible Space Unit - 2026 Defensible Space Cycle\Compliance\2026 Compliance Document Creation\5_28_merged_batches\READY TO PRINT"
)

excel_file = base_folder / "Topanga_Kagel_Search_Results.xlsx"

# Output folders
extract_folder = base_folder / "TOPANGA_KAGEL_EXTRACTED"
revised_folder = base_folder / "REVISED_BATCHES"

extract_folder.mkdir(exist_ok=True)
revised_folder.mkdir(exist_ok=True)

# ==========================================
# READ EXCEL
# ==========================================

df = pd.read_excel(excel_file)

print(f"Rows in spreadsheet: {len(df)}")

# ==========================================
# CREATE MASTER EXTRACTION PDF
# ==========================================

master_extract = fitz.open()

# ==========================================
# GROUP BY PDF
# ==========================================

for pdf_name, group in df.groupby("PDF"):

    source_pdf = base_folder / pdf_name

    if not source_pdf.exists():
        print(f"Missing file: {pdf_name}")
        continue

    print(f"\nProcessing {pdf_name}")

    doc = fitz.open(source_pdf)

    # Convert Excel page numbers to 0-based indexing
    pages_to_remove = sorted(
        set(group["Page"].astype(int) - 1)
    )

    # --------------------------------------
    # EXTRACT MATCHING PAGES
    # --------------------------------------

    for page_num in pages_to_remove:

        temp = fitz.open()

        temp.insert_pdf(
            doc,
            from_page=page_num,
            to_page=page_num
        )

        master_extract.insert_pdf(temp)

        temp.close()

    # --------------------------------------
    # CREATE REVISED PDF
    # --------------------------------------
    
    revised = fitz.open(source_pdf)
    
    # Delete from highest page number to lowest
    for page_num in sorted(pages_to_remove, reverse=True):
        revised.delete_page(page_num)
    
    revised_name = (
        revised_folder /
        pdf_name.replace(".pdf", "_REVISED.pdf")
    )
    
    revised.save(
        revised_name,
        garbage=4,
        deflate=True
    )
    
    revised.close()
    

# ==========================================
# SAVE EXTRACTED PAGES
# ==========================================

master_extract_file = (
    extract_folder /
    "Topanga_Kagel_Extracted.pdf"
)

master_extract.save(master_extract_file)
master_extract.close()

print("\n================================")
print("COMPLETE")
print("================================")
print(f"Extracted pages saved to:")
print(master_extract_file)

print(f"\nRevised PDFs saved to:")
print(revised_folder)